In [1]:
TESTO_DIR = "key_results_testi"

In [2]:
import os
import re
import json
import numpy as np
import pandas as pd
import spacy
from collections import Counter

In [4]:
# Ricarica il modello con i vettori
nlp = spacy.load('en_core_web_md')

OSError: [E050] Can't find model 'en_core_web_md'. It doesn't seem to be a Python package or a valid path to a data directory.

### Dimostrazione del POS Tagging in spaCy

spaCy applica automaticamente il POS tagging (identificazione della parte del discorso) ad ogni token durante l'elaborazione del testo. Possiamo accedere a questa informazione tramite l'attributo `token.pos_`.

In [ ]:
# ============================================================
# FUNZIONE: ESTRAI METADATA DAL NOME FILE
# ============================================================
def estrai_metadata(nome_file):
    base = nome_file.replace("_KeyResults.txt", "")

    # Cerca il pattern mese_anno_-_mese_anno
    periodo_match = re.search(r'([A-Za-z]{3}_\d{4})_-_([A-Za-z]{3}_\d{4})', base)

    if periodo_match:
        inizio_periodo = periodo_match.group(1).replace('_', ' ')  # es. "Apr 2017"
        fine_periodo = periodo_match.group(2).replace('_', ' ')    # es. "Sep 2017"
        periodo = f"{inizio_periodo} / {fine_periodo}"
    else:
        inizio_periodo = None
        fine_periodo = None
        periodo = base

    # Paese: tutto quello che precede il primo mese
    paese_match = re.match(r'^(.+?)_[A-Za-z]{3}_\d{4}', base)
    paese = paese_match.group(1).replace('_', ' ') if paese_match else base

    return {
        "paese": paese,
        "periodo": periodo,
        "inizio_periodo": inizio_periodo,  # es. "Apr 2017"
        "fine_periodo": fine_periodo,       # es. "Sep 2017"
        "nome_file": nome_file
    }

In [ ]:
documents_data = []

for filename in os.listdir(TESTO_DIR):
    file_path = os.path.join(TESTO_DIR, filename)
    if os.path.isfile(file_path) and filename.endswith('.txt'): # Ensure we only process text files
        try:
            # Use the helper function to extract metadata
            metadata = estrai_metadata(filename)

            with open(file_path, 'r', encoding='utf-8') as f:
                file_content = f.read()
                documents_data.append({
                    'filename': filename,
                    'country': metadata['paese'],
                    'period': metadata['periodo'],
                    'text': file_content
                })
            print(f"Read file: {filename}")
        except Exception as e:
            print(f"Error reading {filename}: {e}")

df_documents = pd.DataFrame(documents_data)
print(f"Total documents loaded into DataFrame: {len(df_documents)}")
print("\nFirst 5 rows of the documents DataFrame:")
display(df_documents.head())

Ora leggeremo i documenti dalla directory `TESTO_DIR`. Assumeremo che i file siano in formato testo (`.txt` o simili) e ne estrarremo il contenuto.

In [ ]:
documents_content = []

for filename in os.listdir(TESTO_DIR):
    file_path = os.path.join(TESTO_DIR, filename)
    if os.path.isfile(file_path):
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                documents_content.append(f.read())
            print(f"Read file: {filename}")
        except Exception as e:
            print(f"Error reading {filename}: {e}")

print(f"Total documents loaded: {len(documents_content)}")

Adesso elaboreremo il testo di tutti i documenti usando spaCy per estrarre le parole più comuni. Filtreremo stopword, punteggiatura e numeri, e lemmatizzeremo le parole.

In [ ]:
all_words = []
for doc_text in documents_content:
    doc = nlp(doc_text.lower()) # Process text with spaCy
    # Extract tokens, filter out stopwords, punctuation, and non-alphabetic words, then lemmatize
    words = [token.lemma_ for token in doc if not token.is_stop and not token.is_punct and token.is_alpha]
    all_words.extend(words)

# Count word frequencies
word_counts = Counter(all_words)

# Display the 50 most common words
print("50 parole più comuni nei report:")
for word, count in word_counts.most_common(50):
    print(f"{word}: {count}")

### Analisi delle 50 parole (unigrammi) e dei 50 bigrammi più comuni (con stopword personalizzate)

Questa sezione calcola le frequenze delle parole e dei bigrammi su tutti i documenti combinati, escludendo le stopword di spaCy, la punteggiatura, i non-alfabetici e un set di stopword personalizzate.

In [ ]:
from collections import Counter

# Definisci le tue stopword personalizzate qui
custom_stopwords = {'ipc', 'phase', 'million', 'country', 'level', 'percent', 'classify', 'area', 'analysis', 'period', 'season', 'state'}

all_unigrams = []
all_bigrams = []

for doc_text in documents_content:
    doc = nlp(doc_text.lower()) # Process text with spaCy

    # Filtra e lemmatizza le parole per gli unigrammi e i bigrammi
    filtered_lemmas = [
        token.lemma_
        for token in doc
        if not token.is_stop and not token.is_punct and token.is_alpha and token.lemma_ not in custom_stopwords
    ]
    all_unigrams.extend(filtered_lemmas)

    # Genera bigrammi dalle parole lemmatizzate e filtrate
    bigrams_for_doc = [
        f"{filtered_lemmas[i]} {filtered_lemmas[i+1]}"
        for i in range(len(filtered_lemmas) - 1)
    ]
    all_bigrams.extend(bigrams_for_doc)

# Calcola le frequenze degli unigrammi
unigram_counts = Counter(all_unigrams)

print("--- 50 parole (unigrammi) più comuni (con stopword personalizzate) ---")
for word, count in unigram_counts.most_common(50):
    print(f"{word}: {count}")

print("\n--- 50 bigrammi più comuni (con stopword personalizzate) ---")
bigram_counts = Counter(all_bigrams)
for bigram, count in bigram_counts.most_common(50):
    print(f"{bigram}: {count}")

### Aggiunta Dinamica di Stopword Basate sulla Somiglianza Semantica

Questa sezione implementa un metodo più dinamico per identificare le stopword. Utilizzeremo i vettori di parole di spaCy per calcolare la somiglianza semantica tra le parole più frequenti nel corpus e un concetto chiave (ad esempio, 'crisi alimentare'). Le parole molto distanti semanticamente da questo concetto verranno considerate stopword aggiuntive e rimosse.

**Ricorda:** Se non hai riavviato il runtime dopo aver installato `en_core_web_md` (o `_lg`) e ricaricato `nlp = spacy.load('en_core_web_md')`, la somiglianza potrebbe non funzionare correttamente o i vettori potrebbero mancare.

### Aggiunta Dinamica di Stopword Basate sulla Somiglianza Semantica con POS Tagging

Questa sezione estende il metodo dinamico per identificare le stopword, aggiungendo un filtro per il Part-of-Speech (POS). Prima di calcolare la somiglianza semantica con il concetto chiave, le parole più frequenti verranno filtrate per includere **solo nomi (NOUN)** e **aggettivi (ADJ)**.

Questo assicura che il filtro semantico si applichi a parole che sono concettualmente più significative e meno probabili di essere stopword funzionali.

In [ ]:
#@title Definizione del concetto chiave e della soglia di somiglianza (senza filtro POS sui candidati)

# Definisci i concetti chiave a cui le parole dovrebbero essere semanticamente vicine.
# Puoi usare parole singole o frasi, separate da virgole (es. "food crisis, famine, malnutrition").
CORE_TOPIC_STRING_POS = "food crisis, famine, hunger, conflict, instability, displacement, drought, flood, climate change, economy" #@param {type:"string"}

# Soglia di somiglianza: parole con somiglianza inferiore a questa soglia rispetto a CORE_TOPIC
# verranno considerate semantically distant e aggiunte alle stopword.
# Un valore più alto significa filtrare più parole, un valore più basso filtra meno parole.
SIMILARITY_THRESHOLD_POS = 0.3 #@param {type:"number"}

# Numero di parole più frequenti da considerare per l'analisi di somiglianza
NUM_WORDS_FOR_SIMILARITY_CHECK_POS = 200 #@param {type:"integer"}

# Assicurati che il modello spaCy abbia i vettori di parole
if not nlp.vocab.has_vector: # Verifica se il vocabolario ha vettori
    print("ATTENZIONE: Il modello spaCy caricato non contiene vettori di parole. \n" \
          "Assicurati di aver installato e caricato un modello come 'en_core_web_md' o 'en_core_web_lg' e di aver riavviato il runtime.")
    semantically_distant_stopwords_pos = set()
    all_similarities_pos = []
    word_similarity_map_pos = {}
else:
    # Processa i concetti chiave per ottenere i loro vettori
    core_topics_list_pos = [topic.strip().lower() for topic in CORE_TOPIC_STRING_POS.split(',') if topic.strip()]

    topic_docs_pos = []
    for topic in core_topics_list_pos:
        doc = nlp(topic)
        if doc.has_vector:
            topic_docs_pos.append(doc)
        else:
            print(f"ATTENZIONE: Il concetto chiave '{topic}' non ha un vettore valido. Sarà ignorato.")

    if not topic_docs_pos:
        print("ATTENZIONE: Nessuno concetto chiave valido con vettori è stato trovato. \n" \
              "Impossibile eseguire il filtraggio semantico.")
        semantically_distant_stopwords_pos = set()
        all_similarities_pos = []
        word_similarity_map_pos = {}
    else:
        print(f"Concetti chiave per la somiglianza (senza filtro POS sui candidati): {', '.join(core_topics_list_pos)}")
        print(f"Soglia di somiglianza per le stopword: {SIMILARITY_THRESHOLD_POS}")

        # Prepara una lista delle parole più comuni. NESSUN FILTRO POS qui, per valutare tutte le parole per la distanza semantica.
        candidate_words_for_stopwords_pos = []

        # `word_counts` proviene dalla cella `d4fb7a81` e contiene tutte le parole lemmatizzate
        # `custom_stopwords` proviene dalla cella `e5bf0898`
        if 'custom_stopwords' not in globals():
            initial_custom_stopwords_pos = set()
            print("ATTENZIONE: 'custom_stopwords' non è stata definita. \n" \
                  "Assicurati di aver eseguito la cella che definisce le stopword manuali (e5bf0898).")
        else:
            initial_custom_stopwords_pos = custom_stopwords.copy() # Copia le custom stopwords attuali

        for word_lemma, count in word_counts.most_common(NUM_WORDS_FOR_SIMILARITY_CHECK_POS):
            # Rimuovi il filtro POS qui. Ogni parola frequente è un candidato per il controllo semantico.
            if word_lemma not in initial_custom_stopwords_pos:
                candidate_words_for_stopwords_pos.append(word_lemma)

        semantically_distant_stopwords_pos = set()
        all_similarities_pos = [] # Lista per salvare tutte le somiglianze calcolate
        word_similarity_map_pos = {} # Dizionario per salvare parola -> somiglianza

        print(f"\nControllo somiglianza per le prime {NUM_WORDS_FOR_SIMILARITY_CHECK_POS} parole più frequenti (escluse le stopword iniziali):")
        for word_lemma in candidate_words_for_stopwords_pos:
            word_doc = nlp(word_lemma)
            if word_doc.has_vector:
                max_similarity = max(topic_doc.similarity(word_doc) for topic_doc in topic_docs_pos)
                all_similarities_pos.append(max_similarity) # Aggiungi la somiglianza alla lista
                word_similarity_map_pos[word_lemma] = max_similarity # Aggiungi al dizionario
                if max_similarity < SIMILARITY_THRESHOLD_POS:
                    semantically_distant_stopwords_pos.add(word_lemma)
            # else: # Uncomment for debugging
                # print(f"  - '{word_lemma}' (nessun vettore) -> IGNORATA")

        print(f"\nNumero di stopword aggiunte semanticamente (senza filtro POS sui candidati): {len(semantically_distant_stopwords_pos)}")
        if semantically_distant_stopwords_pos:
            print("Stopword aggiunte dinamicamente (semanticamente distanti):")
            for word in list(semantically_distant_stopwords_pos)[:20]: # Mostra solo le prime 20 per brevità
                print(f"  - {word}")
        else:
            print("Nessuna stopword aggiunta dinamicamente in base alla somiglianza con il/i topic/s.")

# Unisci le stopword personalizzate esistenti con quelle nuove identificate semanticamente
combined_custom_stopwords_pos = initial_custom_stopwords_pos.union(semantically_distant_stopwords_pos)
print(f"\nNumero totale di custom stopword (incluse quelle semanticamente distanti): {len(combined_custom_stopwords_pos)}")

### Filtraggio Automatico di Entità Geografiche e Temporali (NER-based)

In [ ]:
# Definisci le etichette di entità di spaCy che vuoi escludere
# (es. GPE: Geopolitical Entity, LOC: Location, DATE: Dates, TIME: Times)
GEO_TEMPORAL_ENTITY_LABELS = ['GPE', 'LOC', 'DATE', 'TIME']

ner_based_stopwords = set()

print("Identificazione di stopword basate su entità geografiche e temporali...")
for doc_text in documents_content:
    doc = nlp(doc_text.lower())
    for ent in doc.ents:
        if ent.label_ in GEO_TEMPORAL_ENTITY_LABELS:
            ner_based_stopwords.add(ent.lemma_.lower())

print(f"Numero di stopword identificate tramite NER: {len(ner_based_stopwords)}")
if ner_based_stopwords:
    print("Esempi di stopword basate su NER:")
    for word in list(ner_based_stopwords)[:20]: # Mostra solo i primi 20 per brevità
        print(f"  - {word}")

# Unisci tutte le stopword: quelle manuali, quelle semantically distanti e quelle basate su NER
# Assicurati che `combined_custom_stopwords_pos` sia stato definito in una cella precedente (808c8704)
final_filtered_stopwords = combined_custom_stopwords_pos.union(ner_based_stopwords)

print(f"\nNumero totale di custom stopword (manuali + semantiche + NER): {len(final_filtered_stopwords)}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Assicurati che all_similarities_pos sia disponibile
if 'all_similarities_pos' in globals() and len(all_similarities_pos) > 0:
    plt.figure(figsize=(10, 6))
    sns.histplot(all_similarities_pos, bins=20, kde=True)
    plt.title('Distribuzione delle Somiglianze Semantiche delle Parole Candidate')
    plt.xlabel('Valore di Somiglianza')
    plt.ylabel('Frequenza')
    plt.grid(axis='y', alpha=0.75)
    plt.axvline(x=SIMILARITY_THRESHOLD_POS, color='r', linestyle='--', label=f'Soglia Stopword ({SIMILARITY_THRESHOLD_POS:.2f})')
    plt.legend()
    plt.show()
else:
    print("Nessun dato di somiglianza disponibile per la visualizzazione. Assicurati che la cella precedente sia stata eseguita correttamente e che siano stati calcolati i valori di somiglianza.")

### Ricalcolo Unigrammi e Bigrammi con Nuove Stopword Dinamiche

Ora ricalcoleremo le liste di unigrammi e bigrammi utilizzando il set aggiornato di `combined_custom_stopwords` che include anche le parole identificate come semanticamente distanti dal `CORE_TOPIC`.

In [ ]:
from collections import Counter

# Usiamo il nuovo set di stopword combinato che include il filtro POS E le stopword basate su NER
# Assicurati che le celle 808c8704 e fbacfc21 siano state eseguite per definire final_filtered_stopwords
final_stopwords_for_counting = final_filtered_stopwords

all_unigrams_filtered = []
all_bigrams_filtered = []

for doc_text in documents_content:
    doc = nlp(doc_text.lower()) # Process text with spaCy

    # Filtra e lemmatizza le parole per gli unigrammi e i bigrammi
    # Aggiungi il filtro POS qui per includere solo NOMI (NOUN) e AGGETTIVI (ADJ)
    filtered_lemmas_for_final = [
        token.lemma_
        for token in doc
        if not token.is_stop and not token.is_punct and token.is_alpha and token.lemma_ not in final_stopwords_for_counting and token.pos_ in ['NOUN', 'ADJ']
    ]
    all_unigrams_filtered.extend(filtered_lemmas_for_final)

    # Genera bigrammi dalle parole lemmatizzate e filtrate
    bigrams_for_doc_final = [
        f"{filtered_lemmas_for_final[i]} {filtered_lemmas_for_final[i+1]}"
        for i in range(len(filtered_lemmas_for_final) - 1)
    ]
    all_bigrams_filtered.extend(bigrams_for_doc_final)

# Calcola le frequenze degli unigrammi con le stopword dinamiche (ora con filtro POS)
unigram_counts_filtered = Counter(all_unigrams_filtered)

print("--- 50 parole (unigrammi) più comuni (con stopword dinamiche, NER e filtro POS) ---")
for word, count in unigram_counts_filtered.most_common(50):
    print(f"{word}: {count}")

print("\n--- 50 bigrammi più comuni (con stopword dinamiche, NER e filtro POS) ---")
bigram_counts_filtered = Counter(all_bigrams_filtered)
for bigram, count in bigram_counts_filtered.most_common(50):
    print(f"{bigram}: {count}")

### Visualizzazione degli Unigrammi più Comuni: Word Cloud

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

# Ensure unigram_counts_filtered is available
if 'unigram_counts_filtered' in globals() and unigram_counts_filtered:
    # Generate a word cloud image
    wordcloud = WordCloud(width=800, height=400, background_color='white').generate_from_frequencies(unigram_counts_filtered)

    # Display the generated image:
    fig = plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title('Word Cloud of 50 Most Common Unigrams (Filtered with POS)')
    plt.show()
else:
    print("Nessun dato di unigramma filtrato disponibile per la word cloud. Assicurati che la cella precedente sia stata eseguita correttamente.")

### Visualizzazione degli Unigrammi più Comuni: Bubble Word Cloud

Questa visualizzazione alternativa presenta le parole più comuni come 'bolle', dove la dimensione di ciascuna bolla è direttamente proporzionale alla frequenza della parola. Questo approccio offre una rappresentazione più accurata della frequenza rispetto al semplice ridimensionamento del font, che può essere influenzato dalla lunghezza della parola.

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

# Ensure unigram_counts_filtered is available
if 'unigram_counts_filtered' in globals() and unigram_counts_filtered:
    # Get the 50 most common unigrams
    top_50_unigrams_dict = dict(unigram_counts_filtered.most_common(50))

    # Generate a word cloud image
    wordcloud = WordCloud(width=800, height=400, background_color='white').generate_from_frequencies(top_50_unigrams_dict)

    # Display the generated image:
    fig = plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title('Word Cloud of 50 Most Common Unigrams (Filtered with POS)')
    plt.show()
else:
    print("Nessun dato di unigramma filtrato disponibile per la word cloud. Assicurati che la cella precedente sia stata eseguita correttamente.")

### Word Cloud Interattiva per Paese

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from wordcloud import WordCloud
import matplotlib.pyplot as plt
from collections import Counter

def generate_wordcloud_for_country(country_name):
    clear_output(wait=True)

    if country_name == 'Tutti i Paesi':
        # Use all documents if 'All Countries' is selected
        docs_for_wordcloud = df_documents['text']
        title_suffix = "(Tutti i Paesi)"
    else:
        # Filter documents for the selected country
        docs_for_wordcloud = df_documents[df_documents['country'] == country_name]['text']
        title_suffix = f"({country_name})"

    if docs_for_wordcloud.empty:
        print(f"Nessun documento trovato per '{country_name}'.")
        return

    all_unigrams_country = []
    for doc_text in docs_for_wordcloud:
        doc = nlp(doc_text.lower())
        # Apply the same filtering as for the global unigrams (with POS and dynamic stopwords)
        filtered_lemmas = [
            token.lemma_
            for token in doc
            if not token.is_stop and not token.is_punct and token.is_alpha
            and token.lemma_ not in final_filtered_stopwords and token.pos_ in ['NOUN', 'ADJ']
        ]
        all_unigrams_country.extend(filtered_lemmas)

    # Count word frequencies for the selected country
    country_unigram_counts = Counter(all_unigrams_country)

    if not country_unigram_counts:
        print(f"Nessuna parola chiave trovata per '{country_name}' dopo il filtraggio.")
        return

    # Get the 50 most common unigrams for the country
    top_50_country_unigrams_dict = dict(country_unigram_counts.most_common(50))

    # Generate a word cloud image
    wordcloud = WordCloud(width=800, height=400, background_color='white').generate_from_frequencies(top_50_country_unigrams_dict)

    # Display the generated image:
    fig = plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title(f'Word Cloud delle 50 Parole più Comuni {title_suffix}')
    plt.show()

# Get unique country names from the DataFrame and sort them
countries = sorted(df_documents['country'].unique().tolist())
countries.insert(0, 'Tutti i Paesi') # Add an option for all countries

# Create a dropdown widget for country selection
country_selector = widgets.Dropdown(
    options=countries,
    value='Tutti i Paesi', # Default value
    description='Seleziona Paese:',
    disabled=False,
)

# Use interact to link the dropdown to the word cloud generation function
print("Seleziona un paese dal menu a tendina per visualizzare il word cloud:")
widgets.interactive(generate_wordcloud_for_country, country_name=country_selector)


### Word Cloud Interattiva per Paese e Anno

In [ ]:
import re
import ipywidgets as widgets
from IPython.display import display, clear_output
from wordcloud import WordCloud
import matplotlib.pyplot as plt
from collections import Counter

# Step 1: Extract years from the 'period' column and add to df_documents
def extract_years_from_period(period_string):
    years = re.findall(r'\d{4}', period_string)
    return [int(year) for year in years]

# Ensure this column is added only once
if 'years_in_period' not in df_documents.columns:
    df_documents['years_in_period'] = df_documents['period'].apply(extract_years_from_period)

# Function to get available years for a given country
def get_available_years_for_country(country_name):
    if country_name == 'Tutti i Paesi':
        docs_for_years = df_documents
    else:
        docs_for_years = df_documents[df_documents['country'] == country_name]

    if docs_for_years.empty:
        return []

    all_years_for_selection = sorted(list(set(y for years_list in docs_for_years['years_in_period'] for y in years_list)))
    return all_years_for_selection

def generate_wordcloud_for_country_and_year(country_name, selected_year):
    # clear_output(wait=True) # Will be handled by interactive_output

    filtered_docs = df_documents

    # Filter by country
    if country_name != 'Tutti i Paesi':
        filtered_docs = filtered_docs[filtered_docs['country'] == country_name]

    # Filter by year (0 means 'Tutti gli Anni')
    if selected_year != 0:
        filtered_docs = filtered_docs[filtered_docs['years_in_period'].apply(lambda years_list: selected_year in years_list)]

    if filtered_docs.empty:
        year_display = 'tutti gli anni' if selected_year == 0 else f"l'anno '{selected_year}'"
        print(f"Nessun documento trovato per '{country_name}' per {year_display}.")
        return

    all_unigrams_filtered = []
    for doc_text in filtered_docs['text']:
        doc = nlp(doc_text.lower())
        # Apply the same filtering as for the global unigrams (with POS and dynamic stopwords)
        filtered_lemmas = [
            token.lemma_
            for token in doc
            if not token.is_stop and not token.is_punct and token.is_alpha
            and token.lemma_ not in final_filtered_stopwords and token.pos_ in ['NOUN', 'ADJ']
        ]
        all_unigrams_filtered.extend(filtered_lemmas)

    # Count word frequencies for the selected country and year
    country_year_unigram_counts = Counter(all_unigrams_filtered)

    if not country_year_unigram_counts:
        year_display = 'tutti gli anni' if selected_year == 0 else f"l'anno '{selected_year}'"
        print(f"Nessuna parola chiave trovata per '{country_name}' per {year_display} dopo il filtraggio.")
        return

    # Get the 50 most common unigrams
    top_50_country_year_unigrams_dict = dict(country_year_unigram_counts.most_common(50))

    # Generate a word cloud image
    wordcloud = WordCloud(width=800, height=400, background_color='white').generate_from_frequencies(top_50_country_year_unigrams_dict)

    # Display the generated image:
    fig = plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    title_text = f'Word Cloud delle 50 Parole più Comuni per {country_name}'
    if selected_year != 0:
        title_text += f' ({selected_year})'
    else:
        title_text += ' (Tutti gli Anni)'
    plt.title(title_text)
    plt.show()

# Get unique country names from the DataFrame and sort them
countries_options = sorted(df_documents['country'].unique().tolist())
countries_options.insert(0, 'Tutti i Paesi') # Add an option for all countries

# Create a dropdown widget for country selection
country_selector_new = widgets.Dropdown(
    options=countries_options,
    value='Tutti i Paesi', # Default value
    description='Seleziona Paese:',
    disabled=False,
)

# Define the year slider using SelectionSlider for dynamic options
year_selector = widgets.SelectionSlider(
    options=[0], # Initial options: 0 represents 'Tutti gli Anni'
    value=0,     # Default to 'Tutti gli Anni'
    description='Seleziona Anno:',
    disabled=False,
    continuous_update=False # Update only on release for performance
)

# Function to update the year slider's options based on the selected country
def _update_year_slider_range_logic(selected_country):
    available_years = get_available_years_for_country(selected_country)

    if available_years:
        # Sort available years to ensure slider is ordered
        sorted_available_years = sorted(available_years)

        # The options for SelectionSlider should be [0] (for 'Tutti gli Anni') followed by the actual years
        new_options = [0] + sorted_available_years

        year_selector.options = new_options
        year_selector.disabled = False
        year_selector.description = f'Seleziona Anno (Disponibili: {sorted_available_years[0]}-{sorted_available_years[-1]}):'

        # Adjust value if the current value is no longer valid
        if year_selector.value not in new_options:
            year_selector.value = 0 # Default to 'Tutti gli Anni'
    else:
        # If no years are available, only offer 'Tutti gli Anni' (0)
        year_selector.options = [0]
        year_selector.value = 0
        year_selector.disabled = True
        year_selector.description = 'Seleziona Anno (Nessun Anno Disponibile):'

# Wrapper function for the observer
def _on_country_change(change):
    _update_year_slider_range_logic(change.new)

# Attach the observer to the country selector
country_selector_new.observe(_on_country_change, names='value')

# Initial call to set the slider range for the default country selection ("Tutti i Paesi")
# This needs to be called after `year_selector` is defined, and before display.
_update_year_slider_range_logic(country_selector_new.value)

# Use interactive_output to link widgets to the function
out = widgets.interactive_output(
    generate_wordcloud_for_country_and_year,
    {'country_name': country_selector_new, 'selected_year': year_selector}
)

# Display the UI
ui = widgets.VBox([
    widgets.Label("Seleziona un paese dal menu a tendina e un anno dallo slider per visualizzare il word cloud (0 = Tutti gli Anni):"),
    country_selector_new,
    year_selector,
    out
])
display(ui)

### Visualizzazione degli Unigrammi più Comuni: Grafico a Barre

In [ ]:
import seaborn as sns
import pandas as pd

# Ensure unigram_counts_filtered is available
if 'unigram_counts_filtered' in globals() and unigram_counts_filtered:
    # Get the 50 most common unigrams
    top_50_unigrams = unigram_counts_filtered.most_common(50)
    df_top_unigrams = pd.DataFrame(top_50_unigrams, columns=['Word', 'Frequency'])

    # Create the bar chart
    fig = plt.figure(figsize=(12, 8))
    sns.barplot(x='Frequency', y='Word', data=df_top_unigrams, palette='viridis')
    plt.title('50 Most Common Unigrams')
    plt.xlabel('Frequency')
    plt.ylabel('Word')
    plt.tight_layout()
    plt.show()
else:
    print("Nessun dato di unigramma filtrato disponibile per il grafico a barre. Assicurati che la cella precedente sia stata eseguita correttamente.")

### Clustering delle Parole Chiave (Unigrammi) in Macro-Argomenti

Procederemo ora a raggruppare le 50 parole più comuni in macro-argomenti basati sulla loro somiglianza semantica. Utilizzeremo i vettori di parole (word embeddings) forniti dal modello spaCy `en_core_web_md` e l'algoritmo K-Means per il clustering. Successivamente, calcoleremo la frequenza aggregata di ogni macro-argomento.

In [ ]:
from sklearn.cluster import KMeans
import numpy as np
import pandas as pd # Ensure pandas is imported as it's used later

# Assicurati che `df_top_unigrams` e `nlp` siano disponibili
if 'df_top_unigrams' not in globals() or not nlp.vocab.has_vector:
    print("Errore: `df_top_unigrams` non disponibile o modello spaCy senza vettori. ")
    # If `nlp` does not have vectors, it's crucial to inform the user to restart runtime.
    if not nlp.vocab.has_vector:
        print("ATTENZIONE: Il modello spaCy corrente non ha vettori di parole. Si prega di riavviare il runtime dopo aver installato 'en_core_web_md'.")
else:
    # Estrai le parole e i loro vettori
    words = df_top_unigrams['Word'].tolist()
    word_vectors = []
    words_with_vectors = [] # List to store words that actually have vectors

    print("Inizio elaborazione delle parole per il clustering...")
    for word in words:
        doc = nlp(word)
        if len(doc) > 0: # Check if the Doc object contains at least one token
            token = doc[0] # Get the first (and likely only) token from the Doc object
            # Ensure token has a vector and is not punctuation (though should be filtered already)
            if token.has_vector and not token.is_punct:
                word_vectors.append(token.vector)
                words_with_vectors.append(word)
            # else:
            #     print(f"  - Skipping word '{word}': has_vector={token.has_vector}, is_punct={token.is_punct}")
        # else:
        #     print(f"  - Skipping word '{word}': nlp(word) returned an empty Doc.")

    if not word_vectors:
        print("Nessun vettore di parole valido trovato per il clustering. Questo potrebbe essere dovuto a:")
        print("  1. Il modello spaCy caricato ('en_core_web_md') non ha i vettori di parole o non è stato riavviato il runtime.")
        print("  2. Nessuna delle parole più frequenti ha un vettore associato nel modello.")
    else:
        word_vectors_array = np.array(word_vectors)

        # Determina un numero di cluster per le parole (es. 7, puoi ottimizzare questo con il metodo del gomito o silhouette)
        n_word_clusters = 7 # Questo è un valore iniziale, può essere modificato

        print(f"Eseguo K-Means per {n_word_clusters} cluster sulle parole...")
        # Use n_init='auto' to silence future warnings in scikit-learn
        kmeans_words = KMeans(n_clusters=n_word_clusters, random_state=42, n_init='auto')
        kmeans_words.fit(word_vectors_array)

        # Aggiungi le etichette dei cluster alle parole
        # Ensure 'Frequency' is aligned correctly with 'words_with_vectors'
        word_clusters_data = []
        for i, word in enumerate(words_with_vectors):
            # Find the original frequency for this word from df_top_unigrams
            original_freq = df_top_unigrams[df_top_unigrams['Word'] == word]['Frequency'].iloc[0]
            word_clusters_data.append({
                'Word': word,
                'Cluster_Label': kmeans_words.labels_[i],
                'Frequency': original_freq
            })
        word_clusters = pd.DataFrame(word_clusters_data)

        print("\n--- Macro-Argomenti (Cluster di Parole) e Frequenze Aggregate ---")

        # Calcola le frequenze aggregate per ogni macro-argomento
        macro_argument_frequencies = word_clusters.groupby('Cluster_Label')['Frequency'].sum().sort_index()

        for cluster_id in sorted(word_clusters['Cluster_Label'].unique()):
            cluster_words = word_clusters[word_clusters['Cluster_Label'] == cluster_id]
            total_freq = macro_argument_frequencies.loc[cluster_id]
            print(f"\nMacro-Argomento {cluster_id} (Frequenza Totale: {total_freq}):")
            for _, row in cluster_words.sort_values(by='Frequency', ascending=False).iterrows():
                print(f"  - {row['Word']} ({row['Frequency']})")

### Conteggio di Unigrammi e Bigrammi per Ogni Documento

In [ ]:
from collections import Counter

# Usiamo il set di stopword combinato che include quelle manuali, semantiche e basate su NER
# Assicurati che la cella fbacfc21 sia stata eseguita per definire final_filtered_stopwords
final_stopwords_for_doc_analysis = final_filtered_stopwords

top_unigrams_per_doc = []
top_bigrams_per_doc = []

for index, row in df_documents.iterrows():
    doc_text = row['text']
    doc = nlp(doc_text.lower())

    # Filtra e lemmatizza le parole con filtro POS e stopword dinamiche
    filtered_lemmas = [
        token.lemma_
        for token in doc
        if not token.is_stop and not token.is_punct and token.is_alpha and token.lemma_ not in final_stopwords_for_doc_analysis and token.pos_ in ['NOUN', 'ADJ']
    ]

    # Unigrammi
    unigram_counts = Counter(filtered_lemmas)
    top_unigrams_per_doc.append(unigram_counts.most_common(10))

    # Bigrammi
    bigrams_for_doc = [
        f"{filtered_lemmas[i]} {filtered_lemmas[i+1]}"
        for i in range(len(filtered_lemmas) - 1)
    ]
    bigram_counts = Counter(bigrams_for_doc)
    top_bigrams_per_doc.append(bigram_counts.most_common(10))

df_documents['top_unigrams_per_doc'] = top_unigrams_per_doc
df_documents['top_bigrams_per_doc'] = top_bigrams_per_doc

print("DataFrame 'df_documents' aggiornato con unigrammi e bigrammi per documento.")
display(df_documents[['filename', 'country', 'period', 'top_unigrams_per_doc', 'top_bigrams_per_doc']].head())

### Calcolo dei Vettori TF-IDF Basati sulle 50 Parole Globali più Comuni

Ora calcoleremo i vettori TF-IDF per ogni documento, ma ci concentreremo solo sulle 50 parole (unigrammi) più comuni identificate a livello globale. Questo ci permetterà di rappresentare ogni documento in base alla rilevanza dei suoi termini più salienti all'interno del corpus.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Assicurati che `df_top_unigrams` e `unigram_counts_filtered` siano disponibili e aggiornati
# `df_top_unigrams` è stata creata nella cella `227ad5d2`

# Estrai l'elenco delle 50 parole più comuni globali
if 'df_top_unigrams' in globals():
    top_50_words_list = df_top_unigrams['Word'].tolist()
    print(f"Le 50 parole più comuni per TF-IDF sono: {', '.join(top_50_words_list[:10])}...")
else:
    print("Errore: df_top_unigrams non è disponibile. Assicurati che le celle precedenti siano state eseguite.")
    top_50_words_list = [] # Inizializza per evitare errori successivi

# Pre-processa i documenti per includere solo le 50 parole più comuni
preprocessed_docs_for_tfidf = []

for doc_text in df_documents['text']:
    doc = nlp(doc_text.lower())
    # Filtra le parole che sono tra le top 50, mantenendo il loro lemma
    filtered_lemmas_for_tfidf = [
        token.lemma_ for token in doc if token.lemma_ in top_50_words_list
    ]
    preprocessed_docs_for_tfidf.append(' '.join(filtered_lemmas_for_tfidf))

# Inizializza il TfidfVectorizer
# Usiamo il vocabolario predefinito con le nostre 50 parole e un tokenizer semplice
# `lowercase=False` perché abbiamo già convertito in minuscolo e lemmatizzato
tfidf_vectorizer = TfidfVectorizer(
    vocabulary=top_50_words_list,
    tokenizer=lambda text: text.split(), # Le parole sono già spaziate
    lowercase=False
)

# Calcola la matrice TF-IDF
tfidf_matrix = tfidf_vectorizer.fit_transform(preprocessed_docs_for_tfidf)

# Converti la matrice TF-IDF in un DataFrame per una migliore visualizzazione
df_tfidf = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf_vectorizer.get_feature_names_out()
)

print(f"Matrice TF-IDF creata con dimensioni: {df_tfidf.shape}")
print("Prime 5 righe della matrice TF-IDF:")
display(df_tfidf.head())

### Associazione dei Vettori TF-IDF con i Metadati dei Documenti

Come discusso, i vettori TF-IDF sono stati generati nello stesso ordine dei documenti originali. Per unire esplicitamente i vettori TF-IDF con i metadati di ciascun documento, possiamo combinare `df_documents` e `df_tfidf` in un unico DataFrame.

In [ ]:
# Unisci df_documents e df_tfidf
# Poiché entrambi i DataFrame hanno lo stesso indice (implicito, da 0 a N-1) e sono nello stesso ordine,
# possiamo semplicemente concatenarli orizzontalmente.

df_documents_with_tfidf = pd.concat([df_documents, df_tfidf], axis=1)

print("DataFrame combinato 'df_documents_with_tfidf' creato con metadati e feature TF-IDF.")
print(f"Dimensioni del DataFrame combinato: {df_documents_with_tfidf.shape}")
print("Prime 5 righe del DataFrame combinato (solo le prime colonne per chiarezza):")
display(df_documents_with_tfidf.iloc[:, :len(df_documents.columns) + 5]) # Mostra metadati + prime 5 colonne TF-IDF

### Clustering dei Documenti con K-Means

Ora che abbiamo i vettori TF-IDF per ogni documento, possiamo applicare tecniche di clustering per raggruppare i documenti simili. Utilizzeremo l'algoritmo K-Means, che è un metodo popolare per il raggruppamento basato sulla distanza.

### Standardizzazione dei Vettori TF-IDF

Prima di procedere con l'analisi del numero ottimale di cluster, è consigliabile standardizzare i vettori TF-IDF. Questo garantisce che tutte le caratteristiche (le nostre 50 parole più comuni) abbiano un peso simile nel calcolo delle distanze da parte dell'algoritmo di clustering.

In [ ]:
from sklearn.preprocessing import StandardScaler

# `tfidf_features` è già stato estratto nella cella precedente
tfidf_features = df_documents_with_tfidf.iloc[:, 6:]

# Inizializza lo StandardScaler
scaler = StandardScaler()

# Applica la standardizzazione ai vettori TF-IDF
tfidf_features_scaled = scaler.fit_transform(tfidf_features)

print("Vettori TF-IDF standardizzati con successo.")
print(f"Dimensioni dei vettori standardizzati: {tfidf_features_scaled.shape}")

### Determinazione del Numero Ottimale di Cluster: Metodo del Gomito (Elbow Method)

Per trovare un numero adeguato di cluster per l'algoritmo K-Means, useremo il Metodo del Gomito. Questo metodo valuta la *Within-Cluster Sum of Squares* (WCSS) per diversi valori di K. Il 'gomito' nel grafico, dove la diminuzione del WCSS rallenta, suggerisce il numero ottimale di cluster.

In [ ]:
import matplotlib.pyplot as plt

wcss = []
# Prova un intervallo di K, ad esempio da 1 a 50
for i in range(1, 51):
    kmeans = KMeans(n_clusters=i, random_state=42, n_init=10)
    kmeans.fit(tfidf_features_scaled) # Usa i dati standardizzati
    wcss.append(kmeans.inertia_)

# Plotta il risultato
fig = plt.figure(figsize=(10, 6))
plt.plot(range(1, 51), wcss, marker='o')
plt.title('Metodo del Gomito per il K Ottimale')
plt.xlabel('Numero di Cluster (K)')
plt.ylabel('WCSS (Within-Cluster Sum of Squares)')
plt.grid(True)
plt.show()

### Valutazione del Numero Ottimale di Cluster: Coefficiente di Silhouette

Per rafforzare la scelta del numero di cluster, calcoliamo anche il Coefficiente di Silhouette. Questo valore indica la qualità del clustering, misurando quanto ogni oggetto è simile al proprio cluster rispetto ad altri cluster. Valori più alti (vicini a 1) indicano cluster ben separati.

In [ ]:
from sklearn.metrics import silhouette_score

silhouette_avg_scores = []
# Il coefficiente di Silhouette non è definito per un singolo cluster (K=1)
for i in range(2, 51):
    kmeans = KMeans(n_clusters=i, random_state=42, n_init=10)
    kmeans.fit(tfidf_features_scaled)
    score = silhouette_score(tfidf_features_scaled, kmeans.labels_)
    silhouette_avg_scores.append(score)

# Plotta il risultato
fig = plt.figure(figsize=(10, 6))
plt.plot(range(2, 51), silhouette_avg_scores, marker='o')
plt.title('Coefficiente di Silhouette per K Ottimale')
plt.xlabel('Numero di Cluster (K)')
plt.ylabel('Coefficiente di Silhouette Medio')
plt.grid(True)
plt.show()

In [ ]:
from sklearn.cluster import KMeans

# Estrai i vettori TF-IDF dal DataFrame combinato.
# Le colonne TF-IDF iniziano dopo le colonne originali di df_documents.
# Abbiamo 6 colonne originali in df_documents: filename, country, period, text, top_unigrams_per_doc, top_bigrams_per_doc
# Quindi i vettori TF-IDF iniziano dalla settima colonna (indice 6).

tfidf_features = df_documents_with_tfidf.iloc[:, 6:]

# Scegli il numero di cluster (K) in base al Metodo del Gomito.
n_clusters = 40  # Numero di cluster suggerito dal Metodo del Gomito

# Inizializza e addestra il modello KMeans
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10) # n_init per evitare warning su versioni recenti di sklearn
kmeans.fit(tfidf_features_scaled) # Usa i dati standardizzati per il clustering

# Assegna le etichette dei cluster a ogni documento
df_documents_with_tfidf['cluster_label'] = kmeans.labels_

print(f"Clustering K-Means completato con {n_clusters} cluster.")

print("Distribuzione dei documenti per cluster:")
display(df_documents_with_tfidf['cluster_label'].value_counts().sort_index().to_frame())

print("Prime 10 righe del DataFrame con le etichette dei cluster:")
display(df_documents_with_tfidf[['filename', 'country', 'period', 'cluster_label']].head(10))

# ==================================================================
# Analisi dei cluster: parole chiave più importanti per ogni cluster
# ==================================================================
print("\n--- Parole chiave più importanti per ciascun cluster ---")
# Calcola il centroide (media) di ogni cluster per le feature TF-IDF originali (o scalate, ma originali sono più interpretabili)
# Useremo i centroidi del modello KMeans addestrato sui dati scalati.
# Per un'interpretazione più diretta, calcoliamo la media delle feature TF-IDF non scalate per ogni cluster
cluster_centers_df = pd.DataFrame(kmeans.cluster_centers_, columns=tfidf_vectorizer.get_feature_names_out())

# Per ogni cluster, trova le parole con i valori TF-IDF medi più alti
for i, center in cluster_centers_df.iterrows():
    top_words = center.nlargest(10) # Prendi le 10 parole con il valore medio più alto per il cluster
    print(f"\nCluster {i}:")
    for word, score in top_words.items():
        print(f"  - {word}: {score:.3f}")

### Visualizzazione della Distribuzione dei Cluster tramite PCA

Per visualizzare la distribuzione dei cluster in due dimensioni, useremo l'analisi delle componenti principali (PCA) per ridurre la dimensionalità dei vettori TF-IDF scalati. Ogni punto nel grafico rappresenterà un documento, colorato in base al suo cluster assegnato.

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns

# Esegui PCA per ridurre a 2 componenti
pca = PCA(n_components=2)
components = pca.fit_transform(tfidf_features_scaled)

# Crea un DataFrame per la visualizzazione
df_pca = pd.DataFrame(data = components, columns = ['PC1', 'PC2'])
df_pca['cluster_label'] = df_documents_with_tfidf['cluster_label']

# Crea il grafico
fig = plt.figure(figsize=(12, 10))
sns.scatterplot(
    x='PC1',
    y='PC2',
    hue='cluster_label',
    data=df_pca,
    palette=sns.color_palette('tab20', n_colors=n_clusters),
    legend='full',
    alpha=0.7
)
plt.title('Distribuzione dei Cluster (PCA 2D)')
plt.xlabel('Componente Principale 1')
plt.ylabel('Componente Principale 2')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)
plt.tight_layout()
plt.show()

### Analisi del Contenuto dei Documenti nel Cluster 0

Per analizzare il contenuto dei documenti nel Cluster 0, esamineremo i dettagli dei documenti assegnati a questo cluster e calcoleremo gli unigrammi più frequenti al suo interno.

In [ ]:
# Filtra i documenti che appartengono al Cluster 0
cluster_0_docs = df_documents_with_tfidf[df_documents_with_tfidf['cluster_label'] == 0]

print(f"Numero di documenti nel Cluster 0: {len(cluster_0_docs)}")
print("Dettagli dei documenti nel Cluster 0 (prime 10 righe):")
display(cluster_0_docs[['filename', 'country', 'period', 'top_unigrams_per_doc']].head(10))

# Aggrega tutti gli unigrammi dai documenti del Cluster 0
all_unigrams_in_cluster_0 = []
for unigrams_list in cluster_0_docs['top_unigrams_per_doc']:
    # unigrams_list è una lista di tuple (parola, frequenza)
    for word, freq in unigrams_list:
        all_unigrams_in_cluster_0.append(word)

# Conta le frequenze degli unigrammi aggregati
from collections import Counter
unigram_counts_cluster_0 = Counter(all_unigrams_in_cluster_0)

print("\n--- 20 Unigrammi più comuni nel Cluster 0 ---")
for word, count in unigram_counts_cluster_0.most_common(20):
    print(f"{word}: {count}")

### Contenuto di Due Documenti Casuali dal Cluster 0

Per avere un'idea più approfondita dei documenti che compongono il Cluster 0, mostreremo qui il testo completo di due documenti selezionati casualmente da questo cluster.

In [ ]:
# Seleziona due documenti casuali dal cluster_0_docs
sample_docs = cluster_0_docs.sample(n=2, random_state=42)

for index, row in sample_docs.iterrows():
    print(f"\n--- Contenuto del Documento: {row['filename']} ---")
    print(row['text'])
    print("--------------------------------------------------")

### Confronto delle Parole Chiave tra i Due Documenti Campione del Cluster 0

In [ ]:
print("\n--- Confronto Unigrammi ---")
for index, row in sample_docs.iterrows():
    print(f"\nDocumento: {row['filename']}")
    print(f"  Top Unigrammi: {', '.join([word for word, freq in row['top_unigrams_per_doc']])}")

print("\n--- Confronto Bigrammi --- ")
for index, row in sample_docs.iterrows():
    print(f"\nDocumento: {row['filename']}")
    print(f"  Top Bigrammi: {', '.join([bigram for bigram, freq in row['top_bigrams_per_doc']])}")